In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.linear_model import LogisticRegression

# load data
data = pd.read_csv("spambase.data", header=None)

# features and label
X = data.iloc[:, :-1].values
y = data.iloc[:, -1].values

# same 75/25 split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# add bias column manually
X_train_gd = np.c_[np.ones(X_train.shape[0]), X_train]
X_test_gd = np.c_[np.ones(X_test.shape[0]), X_test]

# sigmoid
def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

# cross-entropy loss
def compute_loss(X, y, w):
    p = sigmoid(X @ w)
    eps = 1e-12
    p = np.clip(p, eps, 1 - eps)
    loss = -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))
    return loss

# gradient descent for logistic regression
def logistic_gradient_descent(X, y, learning_rate, iterations):
    n_samples, n_features = X.shape
    w = np.zeros(n_features)
    losses = []

    for i in range(iterations):
        p = sigmoid(X @ w)
        gradient = (X.T @ (p - y)) / n_samples
        w = w - learning_rate * gradient
        loss = compute_loss(X, y, w)
        losses.append(loss)

    return w, losses

# predict probability
def predict_proba_manual(X, w):
    return sigmoid(X @ w)

# predict class
def predict_manual(X, w, threshold=0.5):
    probs = predict_proba_manual(X, w)
    return (probs >= threshold).astype(int)

# evaluate metrics
def evaluate_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    return acc, prec, rec, f1



learning_rates = [0.001, 0.01, 0.1]
results = {}

for lr in learning_rates:
    w, losses = logistic_gradient_descent(X_train_gd, y_train, lr, 100)

    train_pred = predict_manual(X_train_gd, w)
    test_pred = predict_manual(X_test_gd, w)

    train_acc, train_prec, train_rec, train_f1 = evaluate_metrics(y_train, train_pred)
    test_acc, test_prec, test_rec, test_f1 = evaluate_metrics(y_test, test_pred)

    results[lr] = {
        "weights": w,
        "loss_10": losses[9],
        "loss_50": losses[49],
        "loss_100": losses[99],
        "train_metrics": (train_acc, train_prec, train_rec, train_f1),
        "test_metrics": (test_acc, test_prec, test_rec, test_f1)
    }

    print(f"\nLearning Rate = {lr}")
    print("Cross-Entropy Loss after 10 iterations :", losses[9])
    print("Cross-Entropy Loss after 50 iterations :", losses[49])
    print("Cross-Entropy Loss after 100 iterations:", losses[99])

    print("\nMetrics at 100 iterations")
    print("Training Set")
    print("Accuracy :", train_acc)
    print("Precision:", train_prec)
    print("Recall   :", train_rec)
    print("F1 Score :", train_f1)

    print("\nTesting Set")
    print("Accuracy :", test_acc)
    print("Precision:", test_prec)
    print("Recall   :", test_rec)
    print("F1 Score :", test_f1)





Learning Rate = 0.001
Cross-Entropy Loss after 10 iterations : 2.212502492660342
Cross-Entropy Loss after 50 iterations : 1.9849540554029623
Cross-Entropy Loss after 100 iterations: 1.1282554274507626

Metrics at 100 iterations
Training Set
Accuracy : 0.5321739130434783
Precision: 0.4486094316807739
Recall   : 0.8189845474613686
F1 Score : 0.5796875

Testing Set
Accuracy : 0.5438748913987836
Precision: 0.45633456334563344
Recall   : 0.8171806167400881
F1 Score : 0.585635359116022

Learning Rate = 0.01
Cross-Entropy Loss after 10 iterations : 8.853921057632025
Cross-Entropy Loss after 50 iterations : 13.026681759350438
Cross-Entropy Loss after 100 iterations: 13.943739639203645

Metrics at 100 iterations
Training Set
Accuracy : 0.41333333333333333
Precision: 0.40165631469979296
Recall   : 0.9992641648270787
F1 Score : 0.5729957805907172

Testing Set
Accuracy : 0.4135534317984361
Precision: 0.4021257750221435
Recall   : 1.0
F1 Score : 0.5735944409349337

Learning Rate = 0.1
Cross-Entrop

In [2]:
package_model = LogisticRegression(max_iter=10000)
package_model.fit(X_train, y_train)

pkg_train_pred = package_model.predict(X_train)
pkg_test_pred = package_model.predict(X_test)

pkg_train_acc, pkg_train_prec, pkg_train_rec, pkg_train_f1 = evaluate_metrics(y_train, pkg_train_pred)
pkg_test_acc, pkg_test_prec, pkg_test_rec, pkg_test_f1 = evaluate_metrics(y_test, pkg_test_pred)

print("\nPackage Model Metrics")
print("Training Set")
print("Accuracy :", pkg_train_acc)
print("Precision:", pkg_train_prec)
print("Recall   :", pkg_train_rec)
print("F1 Score :", pkg_train_f1)

print("\nTesting Set")
print("Accuracy :", pkg_test_acc)
print("Precision:", pkg_test_prec)
print("Recall   :", pkg_test_rec)
print("F1 Score :", pkg_test_f1)


Package Model Metrics
Training Set
Accuracy : 0.9304347826086956
Precision: 0.9254752851711027
Recall   : 0.8955114054451803
F1 Score : 0.9102468212415857

Testing Set
Accuracy : 0.9296264118158123
Precision: 0.9267734553775744
Recall   : 0.8920704845814978
F1 Score : 0.9090909090909092
